In [2]:
!pip install -q -U langchain langchain-core langchain-community langchain-text-splitters \
    langchain-google-genai langgraph faiss-cpu pypdf gradio pydantic

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
import os
os.environ["GOOGLE_API_KEY"] = "YOUR GEMINI API KEY"

In [4]:
#!/usr/bin/env python3
"""
Agentic Multi-PDF Research Assistant
====================================
Technologies: Python, LangChain, LangGraph, Gemini API, FAISS

What it does
------------
1. Indexes multiple PDFs into a FAISS vector store (RAG).
2. A LangGraph agent runs this workflow:

       plan  ->  retrieve  ->  grade evidence --(sufficient)--> generate answer
                    ^               |
                    |  (insufficient, budget left)
                    +--- follow-up queries

   - plan:      breaks the question into focused search queries
   - retrieve:  FAISS (MMR) search for every pending query, de-duplicated
   - grade:     the LLM judges whether the evidence is enough; if not it
                says what is missing and proposes NEW search queries
   - generate:  answers using only the evidence, with [file.pdf, p.X]
                citations; citations are then verified against what was
                actually retrieved
3. Answers come with a verified "Sources" list (file, page, snippet) and an
   expandable agent trace so users can see how the answer was found.

Install
-------
pip install -U langchain langchain-core langchain-community langchain-text-splitters \
    langchain-google-genai langgraph faiss-cpu pypdf gradio pydantic
# optional, only if EMBEDDING_BACKEND=local:
pip install -U langchain-huggingface sentence-transformers

Run
---
export GOOGLE_API_KEY="your-gemini-key"      # https://aistudio.google.com/apikey
python multi_pdf_research_assistant.py                       # web UI
python multi_pdf_research_assistant.py --pdf-dir ./pdfs \
       --question "Compare the methods used in these papers"  # command line

Optional environment variables
------------------------------
GEMINI_MODEL        default: gemini-2.5-flash
EMBEDDING_BACKEND   gemini (default) | local
GEMINI_EMBED_MODEL  default: models/gemini-embedding-001
"""

from __future__ import annotations

import argparse
import glob
import hashlib
import os
import re
import shutil
import time
from typing import Any, Dict, List, Optional, TypedDict

# --------------------------------------------------------------------------- #
# Auto-install missing dependencies (handy for fresh Colab / Jupyter sessions)
# --------------------------------------------------------------------------- #
def _ensure_dependencies() -> None:
    import importlib
    import importlib.util
    import subprocess
    import sys

    required = {                      # import name -> pip package
        "pydantic": "pydantic",
        "langchain_core": "langchain-core",
        "langchain_community": "langchain-community",
        "langchain_text_splitters": "langchain-text-splitters",
        "langchain_google_genai": "langchain-google-genai",
        "langgraph": "langgraph",
        "faiss": "faiss-cpu",
        "pypdf": "pypdf",
        "gradio": "gradio",
    }
    if os.environ.get("EMBEDDING_BACKEND", "gemini").lower() == "local":
        required["langchain_huggingface"] = "langchain-huggingface"
        required["sentence_transformers"] = "sentence-transformers"

    missing = [pip for mod, pip in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print(f"Installing missing packages: {', '.join(missing)} (one-time, ~1 minute)...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *missing])
        importlib.invalidate_caches()


_ensure_dependencies()

from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph

# --------------------------------------------------------------------------- #
# Configuration
# --------------------------------------------------------------------------- #
PDF_DIR = os.environ.get("PDF_DIR", "./pdfs")
INDEX_DIR = os.environ.get("INDEX_DIR", "./faiss_index")
GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")
EMBEDDING_BACKEND = os.environ.get("EMBEDDING_BACKEND", "gemini").lower()
GEMINI_EMBED_MODEL = os.environ.get("GEMINI_EMBED_MODEL", "models/gemini-embedding-001")

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOP_K = 4              # chunks returned per search query
FETCH_K = 20           # candidate pool for MMR
MAX_QUERIES_PER_STEP = 3
DEFAULT_MAX_ITERATIONS = 3   # max retrieve->grade loops
EMBED_BATCH_SIZE = 64

os.makedirs(PDF_DIR, exist_ok=True)


# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #
def message_text(msg: Any) -> str:
    """Return plain text from a LangChain message (handles list-style content)."""
    content = getattr(msg, "content", msg)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for part in content:
            if isinstance(part, str):
                parts.append(part)
            elif isinstance(part, dict) and part.get("type") == "text":
                parts.append(part.get("text", ""))
        return "".join(parts)
    return str(content)


def with_retry(fn, attempts: int = 5, base_delay: float = 2.0):
    """Retry a callable with exponential backoff (helps with API rate limits)."""
    for i in range(attempts):
        try:
            return fn()
        except Exception:
            if i == attempts - 1:
                raise
            time.sleep(base_delay * (2 ** i))


def format_evidence(docs: List[Document]) -> str:
    blocks = []
    for d in docs:
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", "?")
        blocks.append(f"[{src}, p.{page}]\n{d.page_content}")
    return "\n\n---\n\n".join(blocks)


def doc_key(d: Document) -> str:
    h = hashlib.md5(d.page_content.encode("utf-8", errors="ignore")).hexdigest()[:12]
    return f"{d.metadata.get('source')}|{d.metadata.get('page')}|{h}"


# --------------------------------------------------------------------------- #
# Indexing: load -> chunk -> embed -> FAISS
# --------------------------------------------------------------------------- #
def get_embeddings(api_key: str):
    if EMBEDDING_BACKEND == "local":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return GoogleGenerativeAIEmbeddings(model=GEMINI_EMBED_MODEL, google_api_key=api_key)


def load_pdf_documents(pdf_paths: List[str]) -> List[Document]:
    docs: List[Document] = []
    for path in pdf_paths:
        pages = PyPDFLoader(path).load()
        for p in pages:
            p.metadata["source"] = os.path.basename(path)
            p.metadata["page"] = int(p.metadata.get("page", 0)) + 1  # 1-indexed for humans
        docs.extend(p for p in pages if p.page_content.strip())
    return docs


def build_vectorstore(pdf_paths: List[str], api_key: str, progress=None,
                      existing: Optional[FAISS] = None):
    """Return (vectorstore, n_new_chunks). Embeds in batches to respect rate limits.

    If `existing` is given, the new PDFs are merged into it (incremental indexing).
    """
    report = progress or (lambda frac, desc="": None)

    report(0.05, "Loading PDFs...")
    pages = load_pdf_documents(pdf_paths)
    if not pages:
        raise ValueError(
            "No extractable text found. Scanned/image-only PDFs need OCR first."
        )

    report(0.25, "Splitting into chunks...")
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(pages)

    embeddings = get_embeddings(api_key)
    vectorstore: Optional[FAISS] = existing
    total = len(chunks)
    for start in range(0, total, EMBED_BATCH_SIZE):
        batch = chunks[start:start + EMBED_BATCH_SIZE]
        report(0.3 + 0.65 * (start / total), f"Embedding chunks {start + 1}-{start + len(batch)} of {total}...")
        part = with_retry(lambda b=batch: FAISS.from_documents(b, embeddings))
        if vectorstore is None:
            vectorstore = part
        else:
            vectorstore.merge_from(part)

    os.makedirs(INDEX_DIR, exist_ok=True)
    vectorstore.save_local(INDEX_DIR)
    report(1.0, "Done")
    return vectorstore, total


def load_saved_vectorstore(api_key: str) -> FAISS:
    # allow_dangerous_deserialization is OK here: we only load an index we created.
    return FAISS.load_local(
        INDEX_DIR, get_embeddings(api_key), allow_dangerous_deserialization=True
    )


# --------------------------------------------------------------------------- #
# Structured outputs used by the agent
# --------------------------------------------------------------------------- #
class SearchPlan(BaseModel):
    queries: List[str] = Field(
        description="1-3 short, self-contained search queries that together cover the question."
    )


class EvidenceGrade(BaseModel):
    sufficient: bool = Field(
        description="True only if the evidence fully supports a complete, accurate answer."
    )
    missing_info: str = Field(
        default="", description="What information is still missing (empty if sufficient)."
    )
    follow_up_queries: List[str] = Field(
        default_factory=list,
        description="New, different search queries (max 3) to find the missing information.",
    )


PLAN_PROMPT = ChatPromptTemplate.from_template(
    "You help search a collection of PDF documents.\n"
    "Break the user's question into 1-3 short, self-contained search queries "
    "(keywords/phrases likely to appear in the documents). Use one query if the "
    "question is simple; use several for comparisons or multi-part questions.\n\n"
    "Question: {question}"
)

GRADE_PROMPT = ChatPromptTemplate.from_template(
    "You are a strict evidence checker for a research assistant.\n"
    "Decide whether the EVIDENCE below is sufficient to fully and accurately answer the QUESTION.\n"
    "- sufficient=true only if every part of the question is supported by the evidence.\n"
    "- Otherwise set sufficient=false, describe what is missing, and propose up to 3 NEW "
    "search queries that differ from the ones already tried.\n"
    "- If the documents clearly do not contain the answer after several attempts, "
    "it is acceptable to set sufficient=true and let the answer say so.\n\n"
    "QUESTION: {question}\n\n"
    "QUERIES ALREADY TRIED: {tried}\n\n"
    "EVIDENCE:\n{evidence}"
)

ANSWER_PROMPT = ChatPromptTemplate.from_template(
    "You are a careful research assistant. Answer the question using ONLY the evidence "
    "below, extracted from the user's PDFs.\n\n"
    "Rules:\n"
    "1. Cite every factual statement inline exactly like [filename.pdf, p.X], using only "
    "the file names and page numbers shown in the evidence headers.\n"
    "2. If the evidence only partly answers the question, answer the supported part and "
    "clearly state what the documents do not cover. If it does not answer it at all, say "
    "you could not find it in the documents. Never guess or use outside knowledge.\n"
    "3. When documents disagree, point out the disagreement and cite both.\n"
    "4. Be concise and well organised.\n\n"
    "Evidence status: {status}\n\n"
    "EVIDENCE:\n{evidence}\n\n"
    "QUESTION: {question}\n\n"
    "ANSWER:"
)


# --------------------------------------------------------------------------- #
# LangGraph agent
# --------------------------------------------------------------------------- #
class AgentState(TypedDict, total=False):
    question: str
    pending_queries: List[str]
    all_queries: List[str]
    evidence: List[Document]
    iteration: int
    max_iterations: int
    sufficient: bool
    missing_info: str
    new_chunks_last_step: int
    trace: List[str]
    answer: str
    sources: List[Dict[str, Any]]
    unverified_citations: List[str]


CITATION_RE = re.compile(
    r"([^\[\],;]+?\.pdf)\s*,\s*(?:pages?|pp?)\.?\s*(\d+)", flags=re.IGNORECASE
)


class ResearchAgent:
    """Agentic RAG workflow: plan -> retrieve -> grade -> (loop) -> generate."""

    def __init__(self, vectorstore: FAISS, llm):
        self.vectorstore = vectorstore
        self.llm = llm
        self.planner = PLAN_PROMPT | llm.with_structured_output(SearchPlan)
        self.grader = GRADE_PROMPT | llm.with_structured_output(EvidenceGrade)
        self.answerer = ANSWER_PROMPT | llm
        self.graph = self._build_graph()

    # ---- nodes ---------------------------------------------------------- #
    def plan_node(self, state: AgentState) -> AgentState:
        question = state["question"]
        try:
            plan = self.planner.invoke({"question": question})
            queries = [q.strip() for q in plan.queries if q and q.strip()]
        except Exception:
            queries = []
        if not queries:
            queries = [question]
        queries = queries[:MAX_QUERIES_PER_STEP]
        return {
            "pending_queries": queries,
            "all_queries": [],
            "evidence": [],
            "iteration": 0,
            "trace": [f"**Plan** - search queries: {queries}"],
        }

    def retrieve_node(self, state: AgentState) -> AgentState:
        evidence = list(state.get("evidence", []))
        seen = {doc_key(d) for d in evidence}
        all_queries = list(state.get("all_queries", []))
        new_total = 0
        lines = []

        for q in state.get("pending_queries", []):
            all_queries.append(q)
            try:
                results = self.vectorstore.max_marginal_relevance_search(
                    q, k=TOP_K, fetch_k=FETCH_K
                )
            except Exception as e:  # keep the agent alive on a failed search
                lines.append(f"query '{q}' failed: {e}")
                continue
            added = 0
            for d in results:
                k = doc_key(d)
                if k not in seen:
                    seen.add(k)
                    evidence.append(d)
                    added += 1
            new_total += added
            lines.append(f"'{q}' -> {len(results)} hits, {added} new")

        iteration = state.get("iteration", 0) + 1
        trace = list(state.get("trace", []))
        trace.append(
            f"**Retrieve (round {iteration})** - " + "; ".join(lines)
            + f". Evidence pool: {len(evidence)} chunk(s)."
        )
        return {
            "evidence": evidence,
            "all_queries": all_queries,
            "iteration": iteration,
            "new_chunks_last_step": new_total,
            "pending_queries": [],
            "trace": trace,
        }

    def grade_node(self, state: AgentState) -> AgentState:
        trace = list(state.get("trace", []))
        evidence = state.get("evidence", [])
        if not evidence:
            trace.append("**Grade** - no evidence found yet.")
            sufficient, missing, follow_ups = False, "Nothing relevant retrieved.", []
        else:
            try:
                grade = self.grader.invoke({
                    "question": state["question"],
                    "tried": state.get("all_queries", []),
                    "evidence": format_evidence(evidence),
                })
                sufficient = bool(grade.sufficient)
                missing = grade.missing_info or ""
                follow_ups = [q.strip() for q in grade.follow_up_queries if q and q.strip()]
            except Exception as e:
                # If grading fails, don't loop blindly - proceed to answer.
                sufficient, missing, follow_ups = True, "", []
                trace.append(f"**Grade** - grader error ({e}); proceeding with current evidence.")

        tried = {q.lower() for q in state.get("all_queries", [])}
        follow_ups = [q for q in follow_ups if q.lower() not in tried][:MAX_QUERIES_PER_STEP]

        if evidence and not sufficient:
            trace.append(
                f"**Grade** - evidence NOT sufficient. Missing: {missing or 'unspecified'}. "
                f"Follow-up queries: {follow_ups or 'none'}"
            )
        elif evidence and sufficient:
            trace.append("**Grade** - evidence sufficient.")

        return {
            "sufficient": sufficient,
            "missing_info": missing,
            "pending_queries": follow_ups,
            "trace": trace,
        }

    def generate_node(self, state: AgentState) -> AgentState:
        trace = list(state.get("trace", []))
        evidence = state.get("evidence", [])

        if not evidence:
            return {
                "answer": "I could not find anything relevant to this question in the uploaded documents.",
                "sources": [],
                "unverified_citations": [],
                "trace": trace + ["**Generate** - no evidence; returned a 'not found' answer."],
            }

        if state.get("sufficient"):
            status = "Evidence was judged sufficient."
        else:
            status = (
                "Evidence may be INCOMPLETE after the search budget was used. "
                f"Known gaps: {state.get('missing_info') or 'unspecified'}. "
                "State clearly which parts could not be answered."
            )

        resp = self.answerer.invoke({
            "status": status,
            "evidence": format_evidence(evidence),
            "question": state["question"],
        })
        answer = message_text(resp).strip()

        # ---- verify citations against what was really retrieved ---------- #
        available = {(d.metadata.get("source"), int(d.metadata.get("page", -1))) for d in evidence}
        cited, unverified = [], []
        for name, page in CITATION_RE.findall(answer):
            key = (name.strip(), int(page))
            if key in available:
                if key not in cited:
                    cited.append(key)
            else:
                label = f"{key[0]}, p.{key[1]}"
                if label not in unverified:
                    unverified.append(label)

        sources = []
        chosen = cited if cited else sorted(available, key=lambda x: (str(x[0]), x[1]))
        for (name, page) in chosen:
            snippet = next(
                (d.page_content for d in evidence
                 if d.metadata.get("source") == name and int(d.metadata.get("page", -1)) == page),
                "",
            )
            snippet = " ".join(snippet.split())
            sources.append({
                "file": name,
                "page": page,
                "snippet": snippet[:220] + ("..." if len(snippet) > 220 else ""),
                "cited": bool(cited),
            })

        trace.append(
            f"**Generate** - answered from {len(evidence)} chunk(s); "
            f"{len(cited)} verified citation(s), {len(unverified)} unverified."
        )
        return {
            "answer": answer,
            "sources": sources,
            "unverified_citations": unverified,
            "trace": trace,
        }

    # ---- routing -------------------------------------------------------- #
    @staticmethod
    def route_after_grade(state: AgentState) -> str:
        if state.get("sufficient"):
            return "generate"
        if state.get("iteration", 0) >= state.get("max_iterations", DEFAULT_MAX_ITERATIONS):
            return "generate"           # search budget exhausted
        if not state.get("pending_queries"):
            return "generate"           # nothing new to try
        return "retrieve"               # search again with follow-up queries

    # ---- graph ---------------------------------------------------------- #
    def _build_graph(self):
        g = StateGraph(AgentState)
        g.add_node("plan", self.plan_node)
        g.add_node("retrieve", self.retrieve_node)
        g.add_node("grade", self.grade_node)
        g.add_node("generate", self.generate_node)

        g.add_edge(START, "plan")
        g.add_edge("plan", "retrieve")
        g.add_edge("retrieve", "grade")
        g.add_conditional_edges(
            "grade", self.route_after_grade, {"retrieve": "retrieve", "generate": "generate"}
        )
        g.add_edge("generate", END)
        return g.compile()

    def ask(self, question: str, max_iterations: int = DEFAULT_MAX_ITERATIONS) -> AgentState:
        return self.graph.invoke(
            {"question": question, "max_iterations": int(max_iterations)},
            config={"recursion_limit": 50},
        )


def make_llm(api_key: str, model: str = GEMINI_MODEL) -> ChatGoogleGenerativeAI:
    return ChatGoogleGenerativeAI(
        model=model, google_api_key=api_key, temperature=0.0, max_retries=3
    )


def render_result(result: AgentState) -> str:
    """Format the agent output as markdown: answer + verified sources + trace."""
    out = [result.get("answer", "").strip()]

    sources = result.get("sources", [])
    if sources:
        title = "Sources (cited in the answer)" if sources[0]["cited"] else \
                "Sources (retrieved, not explicitly cited)"
        out.append(f"\n\n**{title}**")
        for s in sources:
            out.append(f"- **{s['file']}**, page {s['page']}: _{s['snippet']}_")

    if result.get("unverified_citations"):
        out.append(
            "\n\n> Warning: these citations could not be matched to retrieved text and "
            "should be double-checked: " + ", ".join(result["unverified_citations"])
        )

    trace = result.get("trace", [])
    if trace:
        out.append("\n\n<details><summary>Agent steps</summary>\n\n"
                   + "\n\n".join(f"{i + 1}. {t}" for i, t in enumerate(trace))
                   + "\n\n</details>")
    return "\n".join(out)


# --------------------------------------------------------------------------- #
# Gradio UI
# --------------------------------------------------------------------------- #
CUSTOM_CSS = """
.gradio-container {max-width: 1280px !important; margin: auto !important;}
footer {display: none !important;}

/* ---------- hero banner ---------- */
.hero {
  background: linear-gradient(120deg, #4338ca 0%, #7c3aed 55%, #0891b2 100%);
  color: #fff; border-radius: 22px; padding: 30px 36px; margin-bottom: 14px;
  box-shadow: 0 12px 32px rgba(79, 70, 229, .28);
}
.hero h1 {margin: 0 0 6px 0; font-size: 2rem; font-weight: 800; color: #fff;}
.hero p  {margin: 0; font-size: 1.02rem; opacity: .92; color: #fff;}
.flow {margin-top: 18px; display: flex; flex-wrap: wrap; align-items: center; gap: 8px;}
.flow .step {
  background: rgba(255,255,255,.17); border: 1px solid rgba(255,255,255,.28);
  padding: 6px 14px; border-radius: 999px; font-size: .85rem; font-weight: 600;
  backdrop-filter: blur(4px);
}
.flow .arrow {opacity: .75; font-weight: 700;}
.tech {margin-top: 14px;}
.tech span {
  display: inline-block; background: rgba(0,0,0,.22); padding: 3px 11px;
  border-radius: 8px; font-size: .75rem; margin: 0 6px 0 0; letter-spacing: .3px;
}

/* ---------- cards ---------- */
.card {
  border: 1px solid var(--border-color-primary) !important;
  border-radius: 18px !important; padding: 16px !important;
  background: var(--background-fill-secondary) !important;
  box-shadow: 0 2px 10px rgba(0,0,0,.05);
}
.card-title {font-size: 1.02rem; font-weight: 700; margin: 0 0 6px 0;}

/* ---------- document chips ---------- */
.doc-count {font-weight: 700; margin-bottom: 8px; color: #4f46e5;}
.doc-chip {
  display: inline-block; margin: 3px 5px 3px 0; padding: 6px 12px; border-radius: 10px;
  background: rgba(99,102,241,.12); border: 1px solid rgba(99,102,241,.35);
  font-size: .82rem; word-break: break-all;
}
.empty-docs {
  text-align: center; padding: 18px 10px; border: 2px dashed var(--border-color-primary);
  border-radius: 14px; opacity: .75; font-size: .92rem;
}

/* ---------- buttons ---------- */
.primary-btn {
  background: linear-gradient(120deg, #4f46e5, #7c3aed) !important; color: #fff !important;
  border: none !important; font-weight: 700 !important; border-radius: 12px !important;
}
.primary-btn:hover {filter: brightness(1.08); transform: translateY(-1px);}
.danger-btn {border-radius: 12px !important;}

/* ---------- chat ---------- */
#chatbot {border-radius: 18px !important;}
#chatbot details {
  margin-top: 10px; padding: 8px 12px; border-radius: 10px;
  background: rgba(99,102,241,.08); border: 1px solid rgba(99,102,241,.25);
}
#chatbot summary {cursor: pointer; font-weight: 600;}
.hint {font-size: .85rem; opacity: .75; margin: 2px 0 0 2px;}
"""

HERO_HTML = """
<div class="hero">
  <h1>📚 Agentic Multi-PDF Research Assistant</h1>
  <p>Ask questions across all your PDFs. The agent searches, checks whether the evidence is
     enough, searches again if it isn't, and answers with verifiable page citations.</p>
  <div class="flow">
    <span class="step">🧭 Plan</span><span class="arrow">→</span>
    <span class="step">🔎 Retrieve</span><span class="arrow">→</span>
    <span class="step">✅ Grade evidence</span><span class="arrow">→</span>
    <span class="step">🔁 Re-search if needed</span><span class="arrow">→</span>
    <span class="step">📝 Answer + citations</span>
  </div>
  <div class="tech"><span>Python</span><span>LangChain</span><span>LangGraph</span>
    <span>Gemini API</span><span>FAISS</span></div>
</div>
"""


def docs_html(names: List[str]) -> str:
    import html
    if not names:
        return ('<div class="empty-docs">📂 No documents indexed yet.<br>'
                'Upload PDFs and click <b>Process Documents</b>.</div>')
    chips = "".join(f'<span class="doc-chip">📄 {html.escape(n)}</span>' for n in names)
    return f'<div class="doc-count">{len(names)} document(s) indexed</div>{chips}'


def launch_ui():
    import gradio as gr

    state: Dict[str, Any] = {
        "vectorstore": None, "agent": None, "agent_key": None, "indexed": [],
    }

    # ------------------------------ handlers ------------------------------ #
    def process_pdfs(files, api_key, progress=gr.Progress()):
        api_key = (api_key or os.environ.get("GOOGLE_API_KEY", "")).strip()
        if not files:
            return "⚠️ Please upload at least one PDF first.", docs_html(state["indexed"])
        if EMBEDDING_BACKEND != "local" and not api_key:
            return "⚠️ Please enter your Gemini API key (Settings).", docs_html(state["indexed"])

        os.makedirs(PDF_DIR, exist_ok=True)
        new_paths, replaced = [], False
        for f in files:
            src = f if isinstance(f, str) else f.name
            name = os.path.basename(src)
            dest = os.path.join(PDF_DIR, name)
            if name in state["indexed"]:
                replaced = True          # same file name uploaded again
            shutil.copy(src, dest)
            new_paths.append(dest)

        report = lambda frac, desc="": progress(frac, desc=desc)
        try:
            if replaced:                 # re-index everything to avoid stale chunks
                keep = {os.path.join(PDF_DIR, n) for n in state["indexed"]} | set(new_paths)
                all_paths = sorted(set(glob.glob(os.path.join(PDF_DIR, "*.pdf"))) & keep)
                vs, n_chunks = build_vectorstore(all_paths, api_key, progress=report)
                state["indexed"] = sorted(os.path.basename(x) for x in all_paths)
            else:                        # add new PDFs to the existing index
                vs, n_chunks = build_vectorstore(
                    new_paths, api_key, progress=report, existing=state["vectorstore"]
                )
                state["indexed"] = sorted(set(state["indexed"]) |
                                          {os.path.basename(x) for x in new_paths})
        except Exception as e:
            return f"❌ Error while indexing: {e}", docs_html(state["indexed"])

        state["vectorstore"] = vs
        state["agent"] = None            # rebuild agent for the updated index
        return (f"✅ Added {n_chunks} chunk(s). You can upload more PDFs and process again.",
                docs_html(state["indexed"]))

    def clear_documents():
        shutil.rmtree(PDF_DIR, ignore_errors=True)
        shutil.rmtree(INDEX_DIR, ignore_errors=True)
        os.makedirs(PDF_DIR, exist_ok=True)
        state.update({"vectorstore": None, "agent": None, "agent_key": None, "indexed": []})
        return "🗑️ All documents removed.", docs_html([]), None

    def respond(message, chat_history, api_key, model_name, max_iter):
        chat_history = chat_history or []
        if not message or not message.strip():
            yield "", chat_history
            return

        chat_history = chat_history + [{"role": "user", "content": message}]
        api_key = (api_key or os.environ.get("GOOGLE_API_KEY", "")).strip()

        if state["vectorstore"] is None:
            reply = "📂 Please upload PDFs and click **Process Documents** first."
        elif not api_key:
            reply = "🔑 Please enter your Gemini API key under **Settings**."
        else:
            # show a "working" bubble while the agent runs
            yield "", chat_history + [{
                "role": "assistant",
                "content": "⏳ *Planning searches, retrieving evidence and checking if it is "
                           "sufficient...*",
            }]
            try:
                key = (api_key, model_name)
                if state["agent"] is None or state["agent_key"] != key:
                    state["agent"] = ResearchAgent(
                        state["vectorstore"], make_llm(api_key, model_name or GEMINI_MODEL)
                    )
                    state["agent_key"] = key
                reply = render_result(state["agent"].ask(message, int(max_iter)))
            except Exception as e:
                reply = f"❌ Error: {e}"

        yield "", chat_history + [{"role": "assistant", "content": reply}]

    def make_chatbot():
        for kw in ({"type": "messages", "show_copy_button": True},
                   {"show_copy_button": True}, {"type": "messages"}, {}):
            try:
                return gr.Chatbot(
                    elem_id="chatbot", height=560, label="Conversation",
                    placeholder="### 👋 Hi!\nUpload your PDFs on the left, then ask me anything "
                                "about them.\n\nI'll show **which file and page** every answer "
                                "comes from.",
                    **kw,
                )
            except TypeError:
                continue
        return gr.Chatbot(height=560)

    # -------------------------------- layout ------------------------------ #
    import inspect
    theme = gr.themes.Soft(
        primary_hue="indigo", secondary_hue="violet", neutral_hue="slate",
        radius_size="lg",
        font=[gr.themes.GoogleFont("Inter"), "system-ui", "sans-serif"],
    )
    # Gradio >= 6 takes theme/css in launch(); Gradio 4/5 take them in Blocks().
    styles_in_launch = "css" in inspect.signature(gr.Blocks.launch).parameters
    if styles_in_launch:
        blocks = gr.Blocks(title="Agentic Multi-PDF Research Assistant")
    else:
        blocks = gr.Blocks(title="Agentic Multi-PDF Research Assistant",
                           theme=theme, css=CUSTOM_CSS)

    with blocks as demo:
        gr.HTML(HERO_HTML)

        with gr.Row(equal_height=False):
            # ---------------- left: documents + settings ---------------- #
            with gr.Column(scale=4, min_width=320):
                with gr.Group(elem_classes="card"):
                    gr.HTML('<div class="card-title">📁 Your documents</div>')
                    file_upload = gr.File(
                        label="Drop PDFs here (Ctrl/Cmd-click to select several)",
                        file_types=[".pdf"], file_count="multiple",
                    )
                    with gr.Row():
                        process_btn = gr.Button("⚡ Process Documents", elem_classes="primary-btn")
                        clear_docs_btn = gr.Button("🗑️ Remove all", elem_classes="danger-btn",
                                                   variant="stop")
                    process_status = gr.Markdown("")
                    docs_panel = gr.HTML(docs_html([]))

                with gr.Accordion("⚙️ Settings", open=True, elem_classes="card"):
                    api_key_box = gr.Textbox(
                        label="Gemini API key", type="password",
                        value=os.environ.get("GOOGLE_API_KEY", ""),
                        placeholder="Paste key or set GOOGLE_API_KEY",
                    )
                    model_box = gr.Textbox(label="Gemini model", value=GEMINI_MODEL)
                    max_iter = gr.Slider(
                        1, 5, value=DEFAULT_MAX_ITERATIONS, step=1,
                        label="Max search rounds",
                        info="How many times the agent may search again when evidence is insufficient.",
                    )

            # ---------------------- right: chat ------------------------- #
            with gr.Column(scale=7, min_width=420):
                with gr.Group(elem_classes="card"):
                    chatbot = make_chatbot()
                    with gr.Row():
                        msg = gr.Textbox(
                            show_label=False, scale=8, lines=1, max_lines=4,
                            placeholder="Ask a question about your documents and press Enter...",
                            container=False,
                        )
                        send_btn = gr.Button("Send ➤", scale=1, elem_classes="primary-btn")
                    gr.Examples(
                        examples=[
                            ["Summarize the key points of each document."],
                            ["Compare the main conclusions across the documents."],
                            ["What methods or approaches are described? Cite the pages."],
                            ["Do any of the documents contradict each other?"],
                        ],
                        inputs=msg, label="💡 Try asking",
                    )
                    with gr.Row():
                        clear = gr.Button("🧹 Clear chat", size="sm")
                    gr.HTML('<div class="hint">Open <b>Agent steps</b> under any answer to see '
                            'the searches, the evidence check and any follow-up searches.</div>')

        # ------------------------------ events ----------------------------- #
        process_btn.click(process_pdfs, inputs=[file_upload, api_key_box],
                          outputs=[process_status, docs_panel])
        clear_docs_btn.click(clear_documents, None,
                             [process_status, docs_panel, file_upload])
        chat_inputs = [msg, chatbot, api_key_box, model_box, max_iter]
        msg.submit(respond, inputs=chat_inputs, outputs=[msg, chatbot])
        send_btn.click(respond, inputs=chat_inputs, outputs=[msg, chatbot])
        clear.click(lambda: [], None, chatbot, queue=False)

    launch_kwargs = dict(share=bool(os.environ.get("GRADIO_SHARE")), debug=True)
    if styles_in_launch:
        launch_kwargs.update(theme=theme, css=CUSTOM_CSS)
    demo.queue().launch(**launch_kwargs)


# --------------------------------------------------------------------------- #
# Command-line mode
# --------------------------------------------------------------------------- #
def run_cli(pdf_dir: str, question: str, max_iterations: int, reuse_index: bool):
    api_key = os.environ.get("GOOGLE_API_KEY", "").strip()
    if not api_key and EMBEDDING_BACKEND != "local":
        raise SystemExit("Set the GOOGLE_API_KEY environment variable first.")

    if reuse_index and os.path.exists(os.path.join(INDEX_DIR, "index.faiss")):
        print(f"Loading saved index from {INDEX_DIR} ...")
        vs = load_saved_vectorstore(api_key)
    else:
        pdfs = sorted(glob.glob(os.path.join(pdf_dir, "*.pdf")))
        if not pdfs:
            raise SystemExit(f"No PDFs found in {pdf_dir}")
        print(f"Indexing {len(pdfs)} PDF(s) ...")
        vs, n = build_vectorstore(pdfs, api_key, progress=lambda f, d="": print(f"  {d}"))
        print(f"Indexed {n} chunks.")

    agent = ResearchAgent(vs, make_llm(api_key))
    print("\n" + render_result(agent.ask(question, max_iterations)))


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Agentic Multi-PDF Research Assistant")
    parser.add_argument("--pdf-dir", default=PDF_DIR, help="Folder with PDFs (CLI mode)")
    parser.add_argument("--question", help="Ask a question from the command line (skips the web UI)")
    parser.add_argument("--max-iterations", type=int, default=DEFAULT_MAX_ITERATIONS)
    parser.add_argument("--reuse-index", action="store_true", help="Load the saved FAISS index instead of re-indexing")
    # parse_known_args so Jupyter/Colab's own "-f kernel.json" argument doesn't crash it
    args, _unknown = parser.parse_known_args()

    if args.question:
        run_cli(args.pdf_dir, args.question, args.max_iterations, args.reuse_index)
    else:
        launch_ui()

/tmp/ipykernel_2543/2458611988.py:94: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Keyboard interruption in main thread... closing server.
